In [1]:
import os

# 必须在所有import之前设置镜像，确保huggingface_hub和sentence-transformers都走镜像
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HUGGINGFACE_HUB_ENDPOINT"] = "https://hf-mirror.com"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = os.path.join(os.path.expanduser("~"), ".cache\huggingface\hub")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

W0320 21:50:13.049000 35512 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## 加载本地模型

In [2]:
# 本地模型路径（镜像已在第一个cell中设置）
QWEN_MODEL_PATH = "Qwen2-0.5B-Instruct"
EMBEDDING_MODEL_NAME = "BAAI/models--BAAI--bge-small-zh-v1.5/snapshots/7999e1d3359715c523056ef9478215996d62a620"

print("正在加载本地 Qwen2-0.5B-Instruct 模型...")
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_PATH,
    torch_dtype=torch.float32,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("Qwen2 模型加载完成！")

print("正在加载本地 Embedding 模型（首次运行会自动从镜像下载）...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding 模型加载完成！")

正在加载本地 Qwen2-0.5B-Instruct 模型...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2 模型加载完成！
正在加载本地 Embedding 模型（首次运行会自动从镜像下载）...


C:\Users\liuqi\AppData\Local\Temp\ipykernel_35512\1812110486.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/models--BAAI--bge-small-zh-v1.5/snapshots/7999e1d3359715c523056ef9478215996d62a620
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 模型加载完成！


## RAG核心逻辑

In [3]:
# 本地Qwen2模型推理函数
def qwen2_chat(prompt_text: str) -> str:
    """使用本地Qwen2-0.5B-Instruct生成回答"""
    messages = [
        {"role": "system", "content": "你是《天天酷跑》游戏专属问答助手，必须严格基于参考资料回答，禁止编造信息。"},
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()

In [4]:
# 加载知识库并构建向量库
def build_ttkp_vector_db():
    # 加载TXT知识库
    loader = TextLoader("data/天天酷跑知识库.txt", encoding="utf-8")
    documents = loader.load()

    # 文档分块（适配游戏文本特点）
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50,
        separators=["\n=== ", "\n", "。", "！", "？", "，"]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"知识库加载完成，共切分 {len(split_docs)} 个文本块")

    # 构建/加载FAISS向量库
    vector_db_path = "ttkp_vector_db_local"
    if os.path.exists(vector_db_path):
        vector_db = FAISS.load_local(vector_db_path, embeddings, allow_dangerous_deserialization=True)
        print("加载已存在的向量库")
    else:
        print("正在向量化（本地模型，不限流）...")
        vector_db = FAISS.from_documents(split_docs, embeddings)
        vector_db.save_local(vector_db_path)
        print("新建向量库并保存")

    return vector_db

In [6]:
# 构建RAG问答链（使用本地Qwen2模型）
def build_rag_chain(vector_db):
    # 定义《天天酷跑》专属Prompt
    prompt_template = """
你是《天天酷跑》游戏专属问答助手，必须严格基于以下参考资料回答，禁止编造任何信息。
参考资料：
{context}

用户问题：{question}

回答要求：
1. 只回答《天天酷跑》相关问题，无关问题直接拒绝回答。
2. 回答简洁明了，优先给出核心结论，再补充关键细节。
3. 涉及数值（如得分加成、冷却时间）必须准确无误。
4. 回答语言口语化，符合游戏玩家的提问习惯。
"""
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["context", "question"]
    )

    # RAG核心逻辑
    def rag_qa(question: str) -> str:
        # 检索Top3相关文本块
        retriever = vector_db.as_retriever(search_kwargs={"k": 3})
        relevant_docs = retriever.invoke(question)
        context = "\n".join([doc.page_content for doc in relevant_docs])

        # 拼接Prompt并调用本地Qwen2
        final_prompt = prompt.format(context=context, question=question)
        return qwen2_chat(final_prompt)

    return rag_qa

## 交互式问答主程序

In [7]:
def main():
    # 构建向量库
    try:
        vector_db = build_ttkp_vector_db()
    except Exception as e:
        print(f"向量库构建失败：{str(e)}")
        return

    # 构建问答链
    rag_qa = build_rag_chain(vector_db)

    # 交互式问答
    print("\n===== 《天天酷跑》RAG知识库问答系统（本地Qwen2模型） =====")
    print("可提问示例：")
    print("   - 孙悟空的技能冷却时间是多少？")
    print("   - 貂蝉搭配什么坐骑得分最高？")
    print("   - 多人对战的胜利条件是什么？")
    print("   - 新手适合用什么角色搭配？")
    print("输入'exit'退出问答\n")

    while True:
        question = input("请输入你的问题：")
        if question.lower() == "exit":
            print("退出问答系统，再见！")
            break
        if not question.strip():
            print("请输入有效问题！")
            continue

        # 生成回答
        print("正在检索并生成回答...")
        answer = rag_qa(question)
        print(f"\n回答：\n{answer}\n" + "-"*50 + "\n")

In [8]:
if __name__ == "__main__":
    main()

知识库加载完成，共切分 8 个文本块
加载已存在的向量库

===== 《天天酷跑》RAG知识库问答系统（本地Qwen2模型） =====
可提问示例：
   - 孙悟空的技能冷却时间是多少？
   - 貂蝉搭配什么坐骑得分最高？
   - 多人对战的胜利条件是什么？
   - 新手适合用什么角色搭配？
输入'exit'退出问答

正在检索并生成回答...

回答：
《天天酷跑》中，貂蝉的技能是“倾城之恋”，全屏吸金，概率无敌。这个技能可以提升貂蝉的得分和金币收益，非常适合喜欢玩策略游戏的玩家。
--------------------------------------------------

正在检索并生成回答...

回答：
《天天酷跑》中，孙悟空的技能时长为12秒/5秒。
--------------------------------------------------

正在检索并生成回答...

回答：
《天天酷跑》中，玩家可以通过多种方式获得积分和奖励，其中一些是通过完成特定任务或挑战来获取的。例如：

- **冷却/持续**：《天天酷跑》支持不同的游戏模式，包括经典模式、进击模式等，这些模式有不同的冷却时间和持续时间。

- **加成**：每个角色都有自己的加成效果，比如表现分加成、飞星加成等。这些加成就根据角色的等级和装备情况而变化。

- **适配**：不同类型的宠物也有不同的适应性，例如芙洛拉可以作为神宠使用，提供额外的加成效果。

请根据以上信息，结合您的实际情况，选择适合您角色的宠物进行搭配。
--------------------------------------------------

退出问答系统，再见！
